# JobMatch AI — Notebook 02: Tokenização com BERTimbau

**Projeto:** JobMatch AI — Sistema de Matching Currículo-Vaga com NLP/Deep Learning
**Autor:** Eduardo Matos
**Etapa:** Tokenização real do BERTimbau aplicada ao dataset completo (N=162)

## O que é tokenização e por que ela importa aqui

Redes neurais não processam texto diretamente — apenas números. Tokenização é o
processo de quebrar o texto em unidades menores (tokens) que o modelo consegue
converter em vetores numéricos.

O BERTimbau usa uma técnica chamada **WordPiece**: palavras comuns viram um único
token; palavras raras ou compostas são quebradas em subpartes (marcadas com `##`
quando são continuação de uma palavra). Isso resolve o problema de vocabulário
"infinito" da linguagem humana com um vocabulário fixo de ~30 mil tokens.

Além disso, o BERT usa tokens especiais:
- `[CLS]`: token inicial, cujo vetor final resume a frase inteira (usado em classificação)
- `[SEP]`: separador entre segmentos de texto
- `[PAD]`: preenchimento para igualar tamanhos dentro de um mesmo lote (batch)
- `[UNK]`: usado quando nem subword resolve (raro, mas indica perda de informação)

## Objetivo deste notebook

1. Carregar o tokenizer oficial do BERTimbau
2. Aplicar a tokenização nas 162 vagas do dataset
3. Medir quantos tokens cada vaga gera, e confirmar que nenhuma excede o limite
   de 512 tokens do BERT (decisão crítica antes do fine-tuning)
4. Identificar quantos tokens `[UNK]` aparecem, como indicador de perda de
   informação por termos fora do vocabulário

In [ ]:
!pip install transformers -q

## 1. Carregando o tokenizer do BERTimbau

Usamos a biblioteca `transformers` (Hugging Face) para baixar o tokenizer oficial
do BERTimbau — modelo BERT pré-treinado especificamente em português pela
Neuralmind/USP.

O tokenizer contém o vocabulário fixo (~30 mil tokens) aprendido durante o
pré-treinamento, junto com as regras de WordPiece para quebrar palavras novas
em subpartes conhecidas.

In [ ]:
from transformers import AutoTokenizer

# BERTimbau - modelo BERT pré-treinado em português (Neuralmind/USP)
NOME_MODELO = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(NOME_MODELO)

print("Tokenizer carregado com sucesso!")
print("Tamanho do vocabulário:", tokenizer.vocab_size)
print("Tokens especiais:", tokenizer.special_tokens_map)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer carregado com sucesso!
Tamanho do vocabulário: 29794
Tokens especiais: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


In [ ]:
import pandas as pd

CAMINHO_CSV = '/content/drive/MyDrive/jobmatch-ai/data/vagas_clean_v2.csv'

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(CAMINHO_CSV)
print('Shape:', df.shape)

# pegando a vaga Getronics ( a mais longa, com termos tecnicos) pra testar
vaga_teste = df.iloc[0]['descricao']
print("\n--- Texto original (primeiros 200 caracteres) ---")
print(vaga_teste[:200])

# Tokenizando
tokens = tokenizer.tokenize(vaga_teste)
print(f"\n--- Total de tokens gerados: {len(tokens)} ---")
print("Primeiros 30 tokens:", tokens[:30])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (162, 13)

--- Texto original (primeiros 200 caracteres) ---
Das 21 milhões de empresas do Brasil, 93% são pequenos e médios negócios e, apesar de serem responsáveis por ⅓ do PIB do país e 60% dos empregos, as pequenas e médias empresas (PMEs) não são o foco da

--- Total de tokens gerados: 98 ---
Primeiros 30 tokens: ['Das', '21', 'milhões', 'de', 'empresas', 'do', 'Brasil', ',', '93', '%', 'são', 'pequenos', 'e', 'médio', '##s', 'negócios', 'e', ',', 'apesar', 'de', 'serem', 'responsáveis', 'por', '[UNK]', 'do', 'PIB', 'do', 'país', 'e', '60']


**Interpretação:** observe os tokens gerados — palavras comuns aparecem inteiras,
enquanto termos técnicos raros ou compostos podem aparecer fragmentados com o
prefixo `##`, indicando continuação de uma palavra anterior (comportamento do
WordPiece).

## 3. Tokenização em lote — todas as 162 vagas

Agora aplicamos o tokenizer em todo o dataset de uma vez, medindo três coisas
para cada vaga:

- **qtd_tokens_com_especiais**: quantidade de tokens incluindo `[CLS]` e `[SEP]`
  — é esse número que precisa ficar dentro do limite de 512 tokens do BERT
- **qtd_unk**: quantos tokens `[UNK]` (desconhecidos) apareceram — indica perda
  de informação por termos fora do vocabulário do BERTimbau
- **excede_512**: sinalizador direto de quais vagas precisariam de truncation

Essa análise é decisiva antes do fine-tuning: se muitas vagas excedessem 512
tokens, precisaríamos definir uma estratégia de corte (truncar do fim, do meio,
etc.). Com o dataset atual, majoritariamente vindo de fontes já truncadas em
500 caracteres, a expectativa é que praticamente nenhuma vaga estoure o limite.

In [ ]:
import matplotlib.pyplot as plt

resultados = []

for idx, row in df.iterrows():
    texto = row['descricao']
    tokens = tokenizer.tokenize(texto)

    # Contagem com tokens especiais (CLS + SEP), que é o que realmente entra no modelo
    tokens_com_especiais = tokenizer(texto)['input_ids']

    qtd_unk = tokens.count('[UNK]')

    resultados.append({
        'titulo': row['titulo'],
        'fit_percentual': row['fit_percentual'],
        'qtd_tokens': len(tokens),
        'qtd_tokens_com_especiais': len(tokens_com_especiais),
        'qtd_unk': qtd_unk,
        'excede_512': len(tokens_com_especiais) > 512
    })

df_tokens = pd.DataFrame(resultados)
print(df_tokens.to_string())

print("\n=== Estatística geral de tokens ===")
print(df_tokens['qtd_tokens_com_especiais'].describe())

print("\n=== Alguma vaga excede 512 tokens? ===")
print(df_tokens['excede_512'].any())

print("\n=== Total de [UNK] no dataset ===")
print(df_tokens['qtd_unk'].sum())

                                                                               titulo  fit_percentual  qtd_tokens  qtd_tokens_com_especiais  qtd_unk  excede_512
0                                                          Fraud Data Analyst - Pleno              75          98                       100        1       False
1                                                                        Data science              60          92                        94        0       False
2                     Pessoa Cientista de Dados - Vaga afirmativa para pessoas negras              80         110                       112        0       False
3                                                Cientista de Dados - Trabalho Remoto              60         229                       231        0       False
4                                                            ciencia de dados pricing              75         120                       122        0       False
5                  Analista De Imp

## 4. Investigando a alta incidência de [UNK]

O total de 148 ocorrências de [UNK] em 162 vagas é desproporcional ao que vimos
na amostra manual original (apenas 1 em 15). Hipótese: o truncamento em 500
caracteres das vagas coletadas via API corta palavras no meio, gerando
fragmentos não reconhecidos pelo vocabulário do BERTimbau.

In [ ]:
# Inspecionando o contexto ao redor de cada [UNK] em algumas vagas
for idx in [0, 20, 50, 80, 113]:  # amostra variada, incluindo o outlier (linha 113, 6 UNKs)
    texto = df.iloc[idx]['descricao']
    tokens = tokenizer.tokenize(texto)

    if '[UNK]' in tokens:
        pos = tokens.index('[UNK]')
        contexto = tokens[max(0, pos-5):pos+5]
        print(f"Vaga {idx} ('{df.iloc[idx]['titulo'][:30]}'): {contexto}")
        print(f"  Fim do texto original: ...{texto[-80:]}")
        print()

Vaga 0 ('Fraud Data Analyst - Pleno'): ['apesar', 'de', 'serem', 'responsáveis', 'por', '[UNK]', 'do', 'PIB', 'do', 'país']
  Fim do texto original: ...clientes e 300 pessoas trabalhando de forma remota em vários lugares do Brasil. 

Vaga 20 ('Cientista de Dados Junior'): ['trabalho', 'na', 'Un', '##ic', '##re', '[UNK]']
  Fim do texto original: ... promove a igualdade nas suas oportunidades! Nosso modelo de trabalho na Unicre…

Vaga 50 ('Analista de BI Júnior'): ['##lig', '##ence', 'atua', 'como', 'pont', '[UNK]']
  Fim do texto original: ...pode ser para você. Sobre a vaga O time de Business Intelligence atua como pont…

Vaga 80 ('Auxiliar Administrativo I - Au'): ['suporte', 'técnico', 'aos', 'usu', '##á', '[UNK]']
  Fim do texto original: ...anilhas mediante a coleta e análise de dados; Fornecer suporte técnico aos usuá…

Vaga 113 ('Assistente financeiro'): ['!', 'Princip', '##ais', 'atividades', ':', '[UNK]', 'Real', '##izar', 'contato', 'com']
  Fim do texto original: ...etor. Re

In [ ]:
# Qual % dos tokens totais são [UNK]? (não a contagem de vagas, mas a proporção real)
total_tokens = df_tokens['qtd_tokens_com_especiais'].sum()
total_unk = df_tokens['qtd_unk'].sum()

print(f"Total de tokens no dataset: {total_tokens}")
print(f"Total de [UNK]: {total_unk}")
print(f"Proporção de [UNK]: {total_unk/total_tokens*100:.2f}%")

Total de tokens no dataset: 18174
Total de [UNK]: 148
Proporção de [UNK]: 0.81%


**Resultado quantitativo:** apenas 0.81% dos tokens do dataset são [UNK]
(148 de 18.174 tokens totais). Essa proporção é considerada aceitável e não
compromete significativamente o treinamento — o modelo perde uma fração
mínima de informação, concentrada majoritariamente no final de textos
truncados pela API (posição de menor peso semântico) ou em caracteres
especiais isolados.

## Conclusões — Tokenização com BERTimbau (N=162)

- Vocabulário do tokenizer: ~29.794 tokens (WordPiece), incluindo os tokens
  especiais [CLS], [SEP], [PAD], [UNK], [MASK]

- Estatística de tokens por vaga: média de 112 tokens, mínimo 8 (vaga com
  descrição muito curta), máximo 231 — nenhuma vaga excede o limite de 512
  tokens do BERT, portanto **não é necessário truncation adicional**

- Taxa de tokens [UNK]: 0.81% do total (148 de 18.174 tokens) — causada
  principalmente por truncamento da fonte de dados (API Adzuna corta em 500
  caracteres, cortando palavras no meio) e, secundariamente, por caracteres
  especiais fora do vocabulário do BERTimbau. Proporção aceitável, sem
  necessidade de tratamento adicional antes do fine-tuning

- Decisão para o fine-tuning: usar max_length=256 (folga confortável acima do
  máximo observado de 231 tokens, evitando desperdício de memória/processamento
  que 512 representaria)

- Próximo passo: gerar embeddings (Notebook 03) e avançar para fine-tuning
  supervisionado do BERTimbau (Notebook 04)